# **Bronze Verification Layer**
Notebook ini digunakan untuk memverifikasi hasil ingestion ke ClickHouse.

Alur verifikasi:
1. Import library
2. Load konfigurasi .env
3. Koneksi ke ClickHouse
4. Cek database dan tabel
5. Validasi row count CSV vs ClickHouse
6. Preview data dan cek schema

#### **1. Import Library**
Import library yang dibutuhkan untuk koneksi, pembacaan CSV, dan query hasil verifikasi.

In [ ]:
import os
import pandas as pd
import clickhouse_connect
from dotenv import load_dotenv

#### **2. Load Konfigurasi**
Ambil konfigurasi dari .env dan siapkan variabel path CSV serta target database/tabel.

In [ ]:
load_dotenv(dotenv_path='../.env')

raw_dir = os.getenv('RAW_DIR')
if not raw_dir:
    raise ValueError('RAW_DIR wajib di-set di .env')

csv_path = os.path.join('..', raw_dir, 'samplesuperstore.csv')
db = "bronze"
table = "bronze_orders"

#### **3. Buka Koneksi ClickHouse**
Gunakan kredensial dari .env untuk membuat client ClickHouse.

In [ ]:
client = clickhouse_connect.get_client(
    host=os.getenv('CH_HOST'),
    port=int(os.getenv('CH_PORT')),
    username=os.getenv('CH_USER'),
    password=os.getenv('CH_PASSWORD'),
)
print('Koneksi ke ClickHouse: OK')

#### **4. Cek Database**
Menampilkan daftar database untuk memastikan koneksi aktif dan database target tersedia.

In [ ]:
df_db = client.query_df('SHOW DATABASES')
print('Databases:')
display(df_db)

#### **5. Cek Tabel Bronze**
Memastikan tabel target di layer bronze sudah terbentuk.

In [ ]:
df_tables = client.query_df(f'SHOW TABLES FROM {db}')
print('Tabel di bronze:')
display(df_tables)

#### **6. Validasi Row Count**
Bandingkan jumlah baris di CSV sumber dengan jumlah baris di tabel ClickHouse.

In [ ]:
# Row count di ClickHouse vs CSV
result = client.query(f'SELECT count() AS row_count FROM {db}.{table}')
ch_count = result.result_rows[0][0]

csv_df = pd.read_csv(csv_path, encoding='utf-8')

csv_count = len(csv_df)

print(f'Row count CSV        : {csv_count:,}')
print(f'Row count ClickHouse : {ch_count:,}')
print()
if ch_count == csv_count:
    print('✅ Row count sesuai — ingestion berhasil!')
else:
    print('❌ Row count TIDAK sesuai — cek ulang proses ingestion.')

#### **7. Preview Data**
Melihat sampel 5 baris pertama dari tabel target untuk sanity check isi data.

In [ ]:
# Preview 5 baris pertama
df_preview = client.query_df(f'SELECT * FROM {db}.{table} LIMIT 5')
print('Preview bronze.bronze_orders (5 baris pertama):')
display(df_preview)

#### **8. Cek Schema Kolom**
Menampilkan nama kolom dan tipe data di ClickHouse untuk verifikasi struktur tabel bronze.

In [ ]:
# Daftar kolom dan tipe data
df_schema = client.query_df(f"""
    SELECT name, type, comment
    FROM system.columns
    WHERE database = '{db}'
      AND table = '{table}'
    ORDER BY position
""")
print(f'Jumlah kolom: {len(df_schema)}')
display(df_schema)